# Environment Check

In [1]:
import sys, os, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', os.getcwd())
!pip install -q --force-reinstall "torch==2.10.0" --index-url https://download.pytorch.org/whl/cu128

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Working directory: /content
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
cuda-python 12.9.7 requires cuda-bindings~=12.9.7, but you have cuda-bindings 12.9.4 which is incompatible.
cupy-cuda12x 14.0.1 requires cuda-pathfinder==1.*,>=1.3.3, but you have cuda-pathfinder 1.2.2 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.10.0+cu128 which is incompatible.
gcsfs 2025

In [2]:
import torch, shutil
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
nvcc = shutil.which('nvcc')
print('nvcc:', nvcc if nvcc else 'not found')

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
nvcc: /usr/local/cuda/bin/nvcc


# Clone Repo

In [3]:
import os
REPO_URL  = 'https://github.com/jeromereddy9/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'
REPO_PATH = '/content/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'
if not os.path.exists(REPO_PATH):
    os.system(f'git clone {REPO_URL}')
os.chdir(REPO_PATH)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

Working directory: /content/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems
Contents: ['.git', '.idea', 'LICENSE', '.gitmodules', '.gitignore', 'src']


# Install Dependencies

In [4]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'numpy==1.26.4', 'recbole==1.2.0'], check=True)
print('recbole installed')

recbole installed


In [5]:
import torch
!pip install -U pip setuptools wheel
!pip install ninja packaging
!pip install causal-conv1d --no-build-isolation
import causal_conv1d
print("causal-conv1d:", causal_conv1d.__version__)
if torch.cuda.is_available():
    print("GPU detected — installing Mamba dependencies...")
    !pip install mamba-ssm==2.3.1 --no-build-isolation
    import mamba_ssm
    print("mamba-ssm version:", mamba_ssm.__version__)
    print("causal-conv1d installed successfully")

else:
    print("No GPU — skipping mamba-ssm")

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 78.1.0
    Uninstalling setuptools-78.1.0:
      Successfully uninstalled setuptools-78.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


causal-conv1d: 1.6.2.post1
GPU detected — installing Mamba dependencies...
mamba-ssm version: 2.3.1
causal-conv1d installed successfully


# Mount Drive and Copy Dataset

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os, shutil

DATASET_SOURCE = '/content/drive/MyDrive/preprocessed/amazon_videogames'
DATASET_ROOT   = os.path.join(REPO_PATH, 'src/datasets/preprocessed')
DATASET_DEST   = os.path.join(DATASET_ROOT, 'amazon_videogames')

os.makedirs(DATASET_ROOT, exist_ok=True)

if not os.path.exists(DATASET_DEST):
    shutil.copytree(DATASET_SOURCE, DATASET_DEST)
    print('Dataset copied.')
else:
    print('Dataset already exists.')

print('Contents:', os.listdir(DATASET_DEST))

Dataset already exists.
Contents: ['amazon_videogames.inter']


# Imports

In [8]:
import sys
sys.path.insert(0, REPO_PATH)

import warnings, logging, traceback
warnings.filterwarnings('ignore')
logging.getLogger('recbole').setLevel(logging.ERROR)

import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

from src.utils import path_builder
from src.models.Baselines.GRU4Rec import GRU4Rec
from src.models.Baselines.SASRec import SASRec
from src.models.Baselines.CL4SRec import CL4SRec
from src.models.Baselines.DouRec import DuoRec
from src.models.Baselines.mamba4rec import Mamba4Rec
from src.models.Baselines.gated_mamba import SIGMA
from src.models.SSM_CL.mamba4rec_cl import Mamba4Rec_CL
from src.models.SSM_CL.SIGMA_cl import SIGMA_CL

print('All imports successful')

All imports successful


# Test Config — SIGMA_CL BPR DCL only

In [12]:
TEST_DATASET      = 'amazon_videogames'
TEST_EPOCHS       = 3
TEST_BATCH        = 256

CONFIG_DIR        = path_builder('src/configs')
DATASET_CONFIG    = path_builder(CONFIG_DIR + '/dataset.yaml')
TRAINING_CONFIG   = path_builder(CONFIG_DIR + '/training.yaml')
MODELS_CONFIG_DIR = path_builder(CONFIG_DIR + '/models')

# Only testing SIGMA_CL BPR DCL
TESTS = [
    (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'BPR'),
]
CL_LOSS_TYPES = ['dcl']

print(f'Test dataset : {TEST_DATASET}')
print(f'Test epochs  : {TEST_EPOCHS}')
print(f'Models       : SIGMA_CL BPR DCL only')

Test dataset : amazon_videogames
Test epochs  : 3
Models       : SIGMA_CL BPR DCL only


# Run Test

In [13]:
import time
def run_test(model_class, model_name, config_file, loss_type, cl_loss_type=None):
    label = model_name
    if loss_type:
        label += f'_{loss_type}'
    if cl_loss_type:
        label += f'_{cl_loss_type}'

    try:
        config_dict = {
            'epochs': TEST_EPOCHS,
            'train_batch_size': TEST_BATCH,
            'eval_batch_size': TEST_BATCH,
            'stopping_step': TEST_EPOCHS,
        }
        if loss_type:
            config_dict['loss_type'] = loss_type
        if cl_loss_type:
            config_dict['cl_loss_type'] = cl_loss_type
        if loss_type == 'BPR':
            config_dict['train_neg_sample_args'] = {
                'distribution': 'uniform',
                'sample_num': 1,
                'alpha': 1.0,
                'dynamic': False,
                'candidate_num': 0
            }

        config = Config(
            model=model_class,
            dataset=TEST_DATASET,
            config_file_list=[
                DATASET_CONFIG,
                TRAINING_CONFIG,
                path_builder(MODELS_CONFIG_DIR + f'/{config_file}.yaml'),
            ],
            config_dict=config_dict,
        )

        config['data_path'] = path_builder('src/datasets/preprocessed/amazon_videogames')

        init_seed(config['seed'], config['reproducibility'])
        init_logger(config)

        print(f'\nRunning: {label}')
        print(f'  Device : {config["device"]}')
        print(f'  GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

        dataset = create_dataset(config)
        train_data, valid_data, test_data = data_preparation(config, dataset)

        model = model_class(config, dataset).to(config['device'])
        trainer = Trainer(config, model)
        start = time.perf_counter()
        trainer.fit(train_data, valid_data, saved=False, show_progress=True)

        elapsed = time.perf_counter() - start

        print(f"\nTraining time: {elapsed / 60:.2f} minutes")
        print(f"Average per epoch: {elapsed / TEST_EPOCHS / 60:.2f} minutes")

        print(f'  PASS  {label}')
        return True

    except Exception as e:
        print(f'  FAIL  {label}')
        print(f'        {e}')
        traceback.print_exc()
        return False


# Run
passed, failed = [], []

for model_class, model_name, config_file, loss_type in TESTS:
    is_cl = model_class in (Mamba4Rec_CL, SIGMA_CL)
    if is_cl:
        for cl_loss_type in CL_LOSS_TYPES:
            ok = run_test(model_class, model_name, config_file, loss_type, cl_loss_type)
            label = f'{model_name}_{loss_type}_{cl_loss_type}'
            (passed if ok else failed).append(label)
    else:
        ok = run_test(model_class, model_name, config_file, loss_type)
        label = f'{model_name}_{loss_type}' if loss_type else model_name
        (passed if ok else failed).append(label)

print(f'\n{"="*40}')
print(f'Passed : {len(passed)}/{len(passed)+len(failed)}')
print(f'Failed : {len(failed)}')
if failed:
    print('Failed tests:')
    for f in failed:
        print(f'  - {f}')


Running: SIGMA_CL_BPR_dcl
  Device : cuda
  GPU    : Tesla T4


Evaluate   : 100%|███████████████████████| 198/198 [00:03<00:00, 50.00it/s, GPU RAM: 1.22 G/14.56 G]



Training time: 14.41 minutes
Average per epoch: 4.80 minutes
  PASS  SIGMA_CL_BPR_dcl

Passed : 1/1
Failed : 0
